In [ ]:
import os
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
def load_data_from_folder(folder_path):
    X = []
    y = []
    for label in os.listdir(folder_path):
        label_path = os.path.join(folder_path, label)
        if os.path.isdir(label_path):
            for file in os.listdir(label_path):
                if file.endswith('.npy'):
                    data = np.load(os.path.join(label_path, file))
                    
                    # THAY ĐỔI Ở ĐÂY: TDÙNG MEAN
                    # Tính trung bình và độ lệch chuẩn theo trục thời gian (axis=1)
                    mean = np.mean(data, axis=1)
                    std = np.std(data, axis=1)
                    features = np.concatenate([mean, std]) # Kết hợp lại
                    
                    X.append(features) 
                    y.append(label)
    return np.array(X), np.array(y)

# Đường dẫn của bạn
train_path = r"C:\Users\Hoannd\Downloads\ail\git_project\-the-adults-cried\data_fine_turn\train"
# x, y = load_data_from_folder(train_path)
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state= 96, stratify= y)
# le = LabelEncoder()
# y_train_encoder = le.fit_transform(y_train)
# y_test_encoder = le.transform(y_test)

test_path = r"C:\Users\Hoannd\Downloads\ail\git_project\-the-adults-cried\data_fine_turn\test"
x_train, y_train = load_data_from_folder(train_path)
x_test, y_test = load_data_from_folder(test_path)
le = LabelEncoder()
y_train_encoder = le.fit_transform(y_train) 
y_test_encoder = le.transform(y_test)



In [ ]:
# print("Total:", len(x))
print("Train:", len(x_train))
print("Test:", len(x_test))

print("\nTrain distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest distribution:")
print(pd.Series(y_test).value_counts())

Total: 2242
Train: 1793
Test: 449

Train distribution:
hungry        593
discomfort    200
belly_pain    200
scared        200
lonely        200
tired         200
burping       200
Name: count, dtype: int64

Test distribution:
hungry        149
tired          50
scared         50
discomfort     50
burping        50
belly_pain     50
lonely         50
Name: count, dtype: int64


## SVC Model 

In [ ]:
pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

param_svm = {
    "svm__kernel": ["linear", "rbf", "poly"],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV(
    pipeline_svm,
    param_svm,
    cv=5,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_svm.fit(x_train, y_train_encoder)

print("Best SVM:", grid_svm.best_params_)
print("Best CV Score:", grid_svm.best_score_)

y_pred_svm = grid_svm.predict(x_test)

print(classification_report(
    y_test_encoder,
    y_pred_svm,
    target_names=le.classes_
))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best SVM: {'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best CV Score: 0.8897236753394824
              precision    recall  f1-score   support

  belly_pain       0.88      0.92      0.90        50
     burping       0.88      0.92      0.90        50
  discomfort       0.78      1.00      0.88        50
      hungry       0.94      0.81      0.87       149
      lonely       0.91      0.96      0.93        50
      scared       0.96      0.94      0.95        50
       tired       0.88      0.90      0.89        50

    accuracy                           0.90       449
   macro avg       0.89      0.92      0.90       449
weighted avg       0.90      0.90      0.89       449



# Random forest 

In [7]:
pipeline_rf = Pipeline([
    ("rf", RandomForestClassifier(random_state=96))
])

param_rf = {
    "rf__n_estimators": [100, 200, 300, 400],
    "rf__criterion": ["gini", "entropy", "log_loss"]
}

grid_rf = GridSearchCV(
    pipeline_rf,
    param_rf,
    cv=5,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_rf.fit(x_train, y_train_encoder)

print("Best Random F:", grid_rf.best_params_)
print("Best CV Score:", grid_rf.best_score_)

y_pred_rf = grid_rf.predict(x_test)

print(classification_report(
    y_test_encoder,
    y_pred_rf,
    target_names=le.classes_
))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Random F: {'rf__criterion': 'gini', 'rf__n_estimators': 200}
Best CV Score: 0.8905028409112289
              precision    recall  f1-score   support

  belly_pain       0.96      0.86      0.91        50
     burping       0.91      0.84      0.88        50
  discomfort       1.00      0.74      0.85        50
      hungry       0.75      1.00      0.86       149
      lonely       1.00      0.82      0.90        50
      scared       1.00      0.86      0.92        50
       tired       0.97      0.76      0.85        50

    accuracy                           0.88       449
   macro avg       0.94      0.84      0.88       449
weighted avg       0.90      0.88      0.88       449



## KNN model


In [8]:
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_knn = {
    "knn__n_neighbors": [3,5,7,9,11],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    pipeline_knn,
    param_knn,
    cv=5,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_knn.fit(x_train, y_train_encoder)

print("Best KNN score:", grid_knn.best_params_)
print("Best CV Score:", grid_knn.best_score_)

y_pred_knn = grid_knn.predict(x_test)

print(classification_report(
    y_test_encoder,
    y_pred_knn,
    target_names=le.classes_
))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best KNN score: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Best CV Score: 0.8381220992718749
              precision    recall  f1-score   support

  belly_pain       0.73      0.86      0.79        50
     burping       0.82      0.94      0.88        50
  discomfort       0.68      1.00      0.81        50
      hungry       0.90      0.64      0.75       149
      lonely       0.89      0.96      0.92        50
      scared       0.89      0.94      0.91        50
       tired       0.85      0.82      0.84        50

    accuracy                           0.83       449
   macro avg       0.82      0.88      0.84       449
weighted avg       0.84      0.83      0.82       449



## XGBOOST

In [ ]:
pipeline_xgb = Pipeline([
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=96
    ))
])

param_xgb = {
    "xgb__n_estimators": [100],
    "xgb__max_depth": [5],
    "xgb__learning_rate": [0.2],
    "xgb__subsample": [1],
    "xgb__colsample_bytree": [1]
}

grid_xgb = GridSearchCV(
    pipeline_xgb,
    param_xgb,
    cv=5,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_xgb.fit(x_train, y_train_encoder)

print("Best XGBoost params:", grid_xgb.best_params_)
print("Best CV Score:", grid_xgb.best_score_)

y_pred_xgb = grid_xgb.predict(x_test)

print(classification_report(
    y_test_encoder, 
    y_pred_xgb,
    target_names=le.classes_
))

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best XGBoost params: {'xgb__colsample_bytree': 1, 'xgb__learning_rate': 0.2, 'xgb__max_depth': 5, 'xgb__n_estimators': 100, 'xgb__subsample': 1}
Best CV Score: 0.8806775358225734
              precision    recall  f1-score   support

  belly_pain       0.94      0.90      0.92        50
     burping       0.98      0.90      0.94        50
  discomfort       0.78      0.80      0.79        50
      hungry       0.86      0.95      0.90       149
      lonely       0.91      0.86      0.89        50
      scared       0.98      0.90      0.94        50
       tired       0.91      0.86      0.89        50

    accuracy                           0.90       449
   macro avg       0.91      0.88      0.89       449
weighted avg       0.90      0.90      0.90       449



## Logistic regression


In [13]:

pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

param_lr = {
    "lr__penalty": ["l1", "l2"],
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__solver": ["liblinear", "saga"]
}

grid_lr = GridSearchCV(
    pipeline_lr,
    param_lr,
    cv=5,
    scoring="f1_macro",
    verbose=2,
    n_jobs=-1
)

grid_lr.fit(x_train, y_train_encoder)

print("Best Logistic params:", grid_lr.best_params_)
print("Best CV Score:", grid_lr.best_score_)

y_pred_lg = grid_lr.predict(x_test)

print(classification_report(
    y_test_encoder,
    y_pred_lg,
    target_names=le.classes_
))

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\Hoannd\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
50 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Hoannd\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Hoannd\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Hoannd\A

Best Logistic params: {'lr__C': 100, 'lr__penalty': 'l1', 'lr__solver': 'saga'}
Best CV Score: 0.4495371792437618
              precision    recall  f1-score   support

  belly_pain       0.67      0.52      0.58        50
     burping       0.52      0.32      0.40        50
  discomfort       0.45      0.34      0.39        50
      hungry       0.46      0.58      0.51       149
      lonely       0.48      0.54      0.51        50
      scared       0.49      0.56      0.52        50
       tired       0.46      0.34      0.39        50

    accuracy                           0.49       449
   macro avg       0.50      0.46      0.47       449
weighted avg       0.49      0.49      0.48       449



In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Vẽ biểu đồ Confusion Matrix trực quan bằng Seaborn Heatmap.
    """
    # Tính toán ma trận
    cm = confusion_matrix(y_true, y_pred)
    
    # Cấu hình biểu đồ
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, 
                yticklabels=class_names)
    
    plt.title('Confusion Matrix ')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# --- LUỒNG CHẠY TIẾP THEO CỦA PIPELINE ---

# Sau khi bạn đã có y_pred và best_model từ code trước:
# Giả sử le là LabelEncoder bạn đã dùng để encode label

print("Đang vẽ Confusion Matrix...")
# plot_confusion_matrix(y_test_encoder, y_thằng nào tự đưa vào đây để vẽ ra, le.classes_)

Đang vẽ Confusion Matrix...
